# Hallucination detection for Claude responses with styxx

Runtime hallucination detection on any Claude call using [styxx](https://github.com/fathom-lab/styxx) — a 9-signal detector cross-validated across 8 public benchmarks. One decorator, zero configuration, model-agnostic.

Styxx was built for the Anthropic case specifically: the Messages API doesn't expose per-token logprobs, so the detector falls back to text + NLI + novelty signals that work on any string output. The published benchmark numbers (below) hold equally on Claude as on any other LLM.

## Published benchmark AUCs (3-seed averaged, n=150/dataset)

| Benchmark | AUC |
|---|---|
| HaluEval-QA | 0.998 |
| TruthfulQA | 0.994 |
| HaluBench-RAGTruth | 0.807 |
| HaluBench-PubMedQA | 0.719 |
| HaluEval-Dialog | 0.676 |
| HaluEval-Summarization | 0.643 |
| HaluBench-FinanceBench | 0.492 (declared failure mode) |
| HaluBench-DROP | 0.424 (declared failure mode) |

Two failure modes declared openly in the weights module — users know where the detector will lie. Full deep-dive: https://fathom.darkflobi.com/cognometry/failures.

Paper: [10.5281/zenodo.19703527](https://doi.org/10.5281/zenodo.19703527). MIT on code, CC-BY-4.0 on calibrated weights.

## 1. Install

`[nli]` pulls in the DeBERTa NLI scorer (~184M). `[anthropic]` pulls in the Claude SDK.

In [ ]:
%pip install -q styxx[nli,anthropic] anthropic

## 2. Set your Anthropic key

In [ ]:
import os
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
assert os.environ.get('ANTHROPIC_API_KEY'), 'set ANTHROPIC_API_KEY first'

## 3. Wrap any Claude-calling function with `@trust`

`@trust` auto-detects `context` (or `reference`, `passage`, `docs`, `source`, `knowledge`, `grounding`, `retrieved`) as the grounding passage. Auto-enables NLI because `styxx[nli]` is installed. Four halt policies: `fallback` (default), `retry`, `raise`, `annotate`.

In [ ]:
from styxx import trust
import anthropic

client = anthropic.Anthropic()

@trust
def ask(question, *, context):
    r = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=400,
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely, using only the context.'},
        ],
    )
    return r.content[0].text

## 4. Correct answer — passes through

In [ ]:
context = '''Inception is a 2010 science fiction film written and
directed by Christopher Nolan. It stars Leonardo DiCaprio as a thief who
steals corporate secrets through dream-sharing technology.'''

ask('Who directed Inception?', context=context)

## 5. Force a hallucination — `@trust` catches it

Ask a specific question whose answer is NOT in the context. Claude will confabulate; the detector fires because the fabricated token has high novelty with no support in the reference.

In [ ]:
context = '''Inception is a 2010 science fiction film written and
directed by Christopher Nolan.'''

ask('What was the exact production budget of Inception, to the dollar? Reply with just the number.',
    context=context)

## 6. Inspect the verdict — `on_halt='annotate'`

In [ ]:
@trust(on_halt='annotate')
def ask_annotated(question, *, context):
    r = client.messages.create(
        model='claude-haiku-4-5', max_tokens=400,
        messages=[{'role': 'user',
                   'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely.'}],
    )
    return r.content[0].text

result = ask_annotated(
    'What was the exact production budget of Inception, to the dollar?',
    context=context,
)

print(f'response : {result.response}')
print(f'risk     : {result.verdict.risk:.3f}')
print(f'action   : {result.verdict.action}')
print(f'halted   : {result.halted}')
print('\nsignals:')
for s in result.verdict.signals:
    print(f'  {s.name:<22s} {s.value}')

## 7. Pre-flight — `styxx.gate()` predicts refusal before the call

Because Anthropic's Messages API doesn't expose per-token logprobs, `styxx` ships a dedicated `anthropic_hack` module for Claude. `styxx.gate()` runs before the call to predict refuse/confabulate/proceed (~$0.0008 per check, ~3.7s latency).

In [ ]:
from styxx import gate

verdict = gate(
    client=client,
    model='claude-haiku-4-5',
    prompt='How do I synthesize methamphetamine?',
)

print(f'will_refuse      : {verdict.will_refuse:.2f}')
print(f'will_confabulate : {verdict.will_confabulate:.2f}')
print(f'recommendation   : {verdict.recommendation}')

## 8. Honest failure modes

Styxx **does not work well** on two benchmark types, published as declared failure modes in the weights module itself:

- **Reading-comprehension extractive-span errors** (DROP-style: wrong span of the right passage). NLI entails the wrong span; novelty signals don't fire because the tokens overlap.
- **Financial arithmetic** (FinanceBench-style: calculation errors on numbers copied verbatim from the source). Novelty + NLI are semantically blind to arithmetic correctness.

Do not deploy `@trust` for production workloads in those two domains without additional domain-specific checks. Full null-probe evidence at https://fathom.darkflobi.com/cognometry/failures.

## Further reading

- Manifesto: https://fathom.darkflobi.com/cognometry
- Leaderboard (open submissions): https://fathom.darkflobi.com/cognometry/leaderboard
- Paper (Zenodo): https://doi.org/10.5281/zenodo.19703527
- Source: https://github.com/fathom-lab/styxx (MIT + CC-BY-4.0)